<a href="https://colab.research.google.com/github/AntonDozhdikov/politpredict/blob/main/BRICS_MARL_SC_Data_Pipeline_v6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BRICS MARL SC — Data Acquisition Pipeline v6



**Итоговый состав — 8 источников, все подтверждены рабочими на 100%:**
OWID, World Bank, WHO GHO, FAOSTAT, Fragile States Index, GDELT, UNGA Voting, BRICS Declarations.

**Запуск:** Runtime → Run all.

In [ ]:
!pip install -q requests pandas numpy pyarrow tqdm openpyxl

In [ ]:
"""
BRICS MARL SC — Data Acquisition & Dataset Assembly Pipeline v6
====================================================================
v6 — ТОЛЬКО реально работающие источники. Удалены/вырезаны полностью:


Итоговый состав — 8 источников, каждый подтверждён рабочим на
реальном прогоне (см. отчёт пользователя от 2026-07-07 23:09):
  OWID, World Bank, WHO GHO, FAOSTAT (bulk ZIP), Fragile States Index,
  GDELT Events, UNGA Voting, BRICS Summit Declarations.

Запуск: python data_pipeline_v6.py
Зависимости: pip install requests pandas numpy pyarrow tqdm openpyxl
"""

import os
import io
import json
import time
import zipfile
import logging
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Callable

import requests
import pandas as pd
import numpy as np



## 0. КОНФИГУРАЦИЯ

In [ ]:

BRICS_ISO3 = ["BRA", "RUS", "IND", "CHN", "ZAF", "EGY", "ETH", "IDN", "IRN", "ARE"]
BRICS_IMF_CODES = {
    "BRA": "223", "RUS": "922", "IND": "534", "CHN": "924",
    "ZAF": "199", "EGY": "469", "ETH": "644", "IDN": "536",
    "IRN": "429", "ARE": "466"
}

DATA_DIR = "data_raw"
OUT_DIR = "data_processed"
LOG_DIR = "logs"
for d in (DATA_DIR, OUT_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(os.path.join(LOG_DIR, "pipeline_v6.log"), encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger("brics_pipeline_v6")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/124.0 Safari/537.36 "
                  "OWID-data-fetch-research/1.0"
}
TIMEOUT = 25
RETRIES = 3
RETRY_SLEEP_BASE = 3




## 1. ИНФРАСТРУКТУРА ИНДИКАЦИИ ЗАГРУЗКИ

In [ ]:

@dataclass
class SourceStatus:
    name: str
    status: str = "not_started"
    rows: int = 0
    cols: int = 0
    countries_covered: int = 0
    date_range: str = ""
    file_path: str = ""
    error: str = ""
    duration_sec: float = 0.0
    notes: str = ""


class LoadReport:
    def __init__(self):
        self.sources: dict = {}

    def start(self, name: str):
        self.sources[name] = SourceStatus(name=name, status="in_progress")
        log.info(f"[{name}] Начало загрузки...")

    def success(self, name, df, file_path="", date_range="", notes="", t0=0.0):
        s = self.sources[name]
        s.status = "ok"
        s.duration_sec = round(time.time() - t0, 2) if t0 else s.duration_sec
        if df is not None and not df.empty:
            s.rows, s.cols = df.shape
            for cc in ("iso3", "code", "country_text_id", "CountryCode"):
                if cc in df.columns:
                    s.countries_covered = df[cc].nunique()
                    break
        s.file_path = file_path
        s.date_range = date_range
        s.notes = notes
        log.info(f"[{name}] OK: {s.rows} строк, {s.cols} колонок, стран={s.countries_covered}")

    def partial(self, name, df, error, t0=0.0):
        s = self.sources[name]
        s.status = "partial"
        s.duration_sec = round(time.time() - t0, 2) if t0 else s.duration_sec
        if df is not None and not df.empty:
            s.rows, s.cols = df.shape
            for cc in ("iso3", "code", "country_text_id", "CountryCode"):
                if cc in df.columns:
                    s.countries_covered = df[cc].nunique()
                    break
        s.error = error
        log.warning(f"[{name}] PARTIAL: {error}")

    def fail(self, name, error, t0=0.0):
        s = self.sources[name]
        s.status = "failed"
        s.duration_sec = round(time.time() - t0, 2) if t0 else s.duration_sec
        s.error = error
        log.error(f"[{name}] FAILED: {error}")

    def print_report(self):
        print("\n" + "=" * 110)
        print("ОТЧЁТ О ЗАГРУЗКЕ ДАННЫХ — BRICS MARL SC PIPELINE v6")
        print(f"Сформирован: {datetime.now().isoformat(timespec='seconds')}")
        print("=" * 110)
        print(f"{'Источник':<42}{'Статус':<10}{'Строк':>9}{'Стран':>8}{'Время(с)':>10}  Комментарий")
        print("-" * 110)
        icon = {"ok": "OK", "partial": "PART", "failed": "FAIL", "not_started": "--", "in_progress": "...", "skipped": "SKIP"}
        for s in self.sources.values():
            comment = s.error if s.error else s.notes
            print(f"{s.name:<42}{icon.get(s.status,'?'):<10}{s.rows:>9}{s.countries_covered:>8}{s.duration_sec:>10.1f}  {comment[:40]}")
        ok = sum(1 for s in self.sources.values() if s.status == "ok")
        part = sum(1 for s in self.sources.values() if s.status == "partial")
        total = len(self.sources)
        print("-" * 110)
        print(f"ИТОГО: успешно {ok}/{total}, частично {part}/{total}")
        print("=" * 110 + "\n")

    def to_json(self, path=os.path.join(LOG_DIR, "load_report_v6.json")):
        data = {k: vars(v) for k, v in self.sources.items()}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        log.info(f"Отчёт сохранён в {path}")


REPORT = LoadReport()


def with_retries(fn: Callable, name: str, retries: int = RETRIES):
    last_exc = None
    for attempt in range(1, retries + 1):
        try:
            return fn()
        except Exception as e:
            last_exc = e
            sleep_s = RETRY_SLEEP_BASE * (2 ** (attempt - 1))
            log.warning(f"[{name}] попытка {attempt}/{retries} не удалась: {e}. Ждём {sleep_s}с")
            if attempt < retries:
                time.sleep(sleep_s)
    raise last_exc


def safe_get(url, params=None, timeout=TIMEOUT):
    r = requests.get(url, params=params, headers=HEADERS, timeout=timeout)
    r.raise_for_status()
    return r




## 2. OUR WORLD IN DATA — прямые CSV

In [ ]:

OWID_DATASETS = {
    "military_spending_gdp": "military-expenditure-as-a-share-of-gdp",
    "military_spending_usd": "military-spending-sipri",
    "gdp_per_capita": "gdp-per-capita-worldbank",
    "gdp_growth": "gdp-per-capita-growth",
    "population": "population",
    "population_growth_rate": "population-growth-rate",
    "trade_openness": "trade-openness",
    "internet_users_pct": "share-of-individuals-using-the-internet",
    "urbanization": "share-of-population-urban",
    "life_expectancy": "life-expectancy",
    "co2_emissions": "annual-co2-emissions-per-country",
    "press_freedom": "press-freedom-index-rsf",
    "electoral_democracy_vdem": "electoral-democracy-index",
    "liberal_democracy_vdem": "liberal-democracy-index",
    "political_corruption_vdem": "political-corruption-index",
    "corruption_perception": "ti-corruption-perception-index",
    "unemployment_rate": "unemployment-rate",
    "gini_index": "economic-inequality-gini-index",
    "fdi_inflows": "foreign-direct-investment-net-inflows-of-gdp",
    "electricity_access": "share-of-the-population-with-access-to-electricity",
    "research_dev_spending": "research-and-development-expenditure-of-gdp",
    "human_development_index": "human-development-index",
    "food_supply_kcal": "daily-per-capita-caloric-supply",
}


def fetch_owid_dataset(slug: str) -> pd.DataFrame:
    url = f"https://ourworldindata.org/grapher/{slug}.csv"
    params = {"v": "1", "csvType": "full", "useColumnShortNames": "true"}

    def _get():
        r = safe_get(url, params=params, timeout=TIMEOUT)
        return r.content

    content = with_retries(_get, f"OWID:{slug}", retries=3)
    df = pd.read_csv(io.BytesIO(content))
    df.columns = [c.strip() for c in df.columns]
    return df


def fetch_all_owid() -> pd.DataFrame:
    name = "Our World in Data (индикаторы)"
    REPORT.start(name)
    t0 = time.time()
    frames = []
    ok_slugs, failed_slugs = [], []

    for alias, slug in OWID_DATASETS.items():
        try:
            df = fetch_owid_dataset(slug)
            cols_lower = {c.lower(): c for c in df.columns}
            id_col = cols_lower.get("code") or cols_lower.get("iso_code")
            year_col = cols_lower.get("year")
            entity_col = cols_lower.get("entity")

            if id_col is None or year_col is None:
                failed_slugs.append(f"{alias}(нет code/year)")
                continue

            exclude = {id_col, year_col, entity_col} - {None}
            value_cols = [c for c in df.columns if c not in exclude]
            if not value_cols:
                failed_slugs.append(f"{alias}(нет колонки значения)")
                continue

            sub = df[[id_col, year_col, value_cols[0]]].copy()
            sub.columns = ["iso3", "year", alias]
            sub = sub[sub["iso3"].isin(BRICS_ISO3)]
            sub["year"] = pd.to_numeric(sub["year"], errors="coerce")
            sub[alias] = pd.to_numeric(sub[alias], errors="coerce")

            if sub.empty:
                failed_slugs.append(f"{alias}(0 строк после фильтра BRICS)")
                continue

            frames.append(sub)
            ok_slugs.append(alias)
            time.sleep(0.25)
        except Exception as e:
            failed_slugs.append(f"{alias}({str(e)[:35]})")
            log.debug(f"[{name}] {alias} ({slug}) не загружен: {e}")

    if not frames:
        REPORT.fail(name, f"0/{len(OWID_DATASETS)} загружено. Ошибки: {failed_slugs[:6]}", t0)
        return pd.DataFrame()

    merged = frames[0]
    for f_ in frames[1:]:
        merged = merged.merge(f_, on=["iso3", "year"], how="outer")

    path = os.path.join(DATA_DIR, "owid_merged.parquet")
    merged.to_parquet(path, index=False)

    notes = f"{len(ok_slugs)}/{len(OWID_DATASETS)} индикаторов загружено"
    if failed_slugs:
        REPORT.partial(name, merged, f"{notes}; не загружены: {failed_slugs[:5]}", t0)
    else:
        REPORT.success(name, merged, path, notes=notes, t0=t0)
    return merged




## 3. WORLD BANK API

In [ ]:

WB_INDICATORS = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_pct",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_inflow_pct_gdp",
    "SP.POP.TOTL": "population_total",
    "SP.POP.DPND": "dependency_ratio",
    "SI.POV.GINI": "gini_index_wb",
    "SL.UEM.TOTL.ZS": "unemployment_pct_wb",
    "MS.MIL.XPND.GD.ZS": "military_exp_pct_gdp_wb",
    "SM.POP.NETM": "net_migration_wb",
    "IT.NET.USER.ZS": "internet_users_pct_wb",
    "NY.GDP.PCAP.CD": "gdp_per_capita_usd",
    "FP.CPI.TOTL.ZG": "inflation_pct",
    "GC.DOD.TOTL.GD.ZS": "gov_debt_pct_gdp",
    "SP.URB.TOTL.IN.ZS": "urban_pop_pct",
    "SE.XPD.TOTL.GD.ZS": "education_exp_pct_gdp",
}


def fetch_wb_single(iso3: str, code: str, start_year=1995, end_year=2024) -> pd.DataFrame:
    url = f"https://api.worldbank.org/v2/country/{iso3}/indicator/{code}"
    params = {"date": f"{start_year}:{end_year}", "format": "json", "per_page": "1000"}

    def _get():
        r = safe_get(url, params=params, timeout=20)
        return r.json()

    try:
        js = with_retries(_get, f"WB:{iso3}:{code}", retries=2)
    except Exception:
        def _get_fallback():
            r = safe_get(url, params={"format": "json", "per_page": "1000"}, timeout=20)
            return r.json()
        js = with_retries(_get_fallback, f"WB-fallback:{iso3}:{code}", retries=2)

    if not isinstance(js, list) or len(js) < 2 or js[1] is None:
        return pd.DataFrame()
    recs = js[1]
    if not recs:
        return pd.DataFrame()
    df = pd.DataFrame(recs)
    df["iso3"] = df["countryiso3code"]
    df["year"] = pd.to_numeric(df["date"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    return df[["iso3", "year", "value"]]


def fetch_world_bank() -> pd.DataFrame:
    name = "World Bank Open Data (по странам)"
    REPORT.start(name)
    t0 = time.time()
    all_frames = []
    fail_pairs = []

    for code, alias in WB_INDICATORS.items():
        indicator_frames = []
        for iso3 in BRICS_ISO3:
            try:
                sub = fetch_wb_single(iso3, code)
                if not sub.empty:
                    sub = sub.rename(columns={"value": alias})
                    indicator_frames.append(sub)
                else:
                    fail_pairs.append(f"{iso3}:{alias}(empty)")
            except Exception as e:
                fail_pairs.append(f"{iso3}:{alias}")
                log.debug(f"[{name}] {iso3}/{alias} failed: {e}")
        if indicator_frames:
            all_frames.append(pd.concat(indicator_frames, ignore_index=True))

    if not all_frames:
        REPORT.fail(name, "Ни один индикатор не загружен ни для одной страны", t0)
        return pd.DataFrame()

    merged = all_frames[0]
    for f_ in all_frames[1:]:
        merged = merged.merge(f_, on=["iso3", "year"], how="outer")

    path = os.path.join(DATA_DIR, "world_bank_v5.parquet")
    merged.to_parquet(path, index=False)

    n_ok = len(all_frames)
    notes = f"{n_ok}/{len(WB_INDICATORS)} индикаторов; неудачных пар: {len(fail_pairs)}"
    if fail_pairs:
        REPORT.partial(name, merged, notes, t0)
    else:
        REPORT.success(name, merged, path, "1995-2024", notes, t0)
    return merged




## 4. DBNOMICS (зеркало IMF DOT)

In [ ]:
# 5. GDELT
# ----------------------------------------------------------------------------

GDELT_COLS = [
    "GLOBALEVENTID", "SQLDATE", "MonthYear", "Year", "FractionDate",
    "Actor1Code", "Actor1Name", "Actor1CountryCode",
    "Actor2Code", "Actor2Name", "Actor2CountryCode",
    "IsRootEvent", "EventCode", "EventBaseCode", "EventRootCode",
    "QuadClass", "GoldsteinScale", "NumMentions", "NumSources",
    "NumArticles", "AvgTone", "Actor1Geo_CountryCode", "Actor2Geo_CountryCode",
    "DATEADDED", "SOURCEURL"
]

def fetch_gdelt_events(days_sample):
    name = "GDELT Events (выборка ключевых дат)"
    REPORT.start(name)
    t0 = time.time()
    frames = []
    failed_days = []

    for day in days_sample:
        url = f"http://data.gdeltproject.org/events/{day}.export.CSV.zip"
        try:
            def _get():
                r = safe_get(url, timeout=40)
                return r.content
            content = with_retries(_get, f"{name}:{day}", retries=2)
            zf = zipfile.ZipFile(io.BytesIO(content))
            fname = zf.namelist()[0]
            raw = zf.read(fname).decode("utf-8", errors="ignore")
            df = pd.read_csv(io.StringIO(raw), sep="\t", header=None, low_memory=False)
            df = df.iloc[:, :len(GDELT_COLS)]
            df.columns = GDELT_COLS[:df.shape[1]]
            mask = (df["Actor1CountryCode"].isin(BRICS_ISO3) | df["Actor2CountryCode"].isin(BRICS_ISO3))
            frames.append(df[mask])
        except Exception as e:
            failed_days.append(day)
            log.debug(f"[{name}] {day} не загружен: {e}")

    if not frames:
        REPORT.fail(name, f"Не удалось загрузить ни один из {len(days_sample)} дней", t0)
        return pd.DataFrame()

    full = pd.concat(frames, ignore_index=True)
    path = os.path.join(DATA_DIR, "gdelt_events_sample.parquet")
    full.to_parquet(path, index=False)

    notes = f"{len(frames)}/{len(days_sample)} дней загружено"
    if failed_days:
        REPORT.partial(name, full, f"{notes}; неудачные: {failed_days}", t0)
    else:
        REPORT.success(name, full, path, notes=notes, t0=t0)
    return full




## 6. UN GENERAL ASSEMBLY VOTING

In [ ]:

def fetch_unga_votes() -> pd.DataFrame:
    name = "UNGA Voting"
    REPORT.start(name)
    t0 = time.time()
    candidate_urls = [
        "https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2021/2021-03-23/unvotes.csv",
        "https://raw.githubusercontent.com/dgrtwo/unvotes/master/data-raw/votes.csv",
    ]
    for url in candidate_urls:
        try:
            def _get():
                r = safe_get(url, timeout=40)
                return r.content
            content = with_retries(_get, name, retries=2)
            df = pd.read_csv(io.BytesIO(content), low_memory=False)
            path = os.path.join(DATA_DIR, "unga_votes.parquet")
            df.to_parquet(path, index=False)
            REPORT.success(name, df, path, notes=f"источник: {url}", t0=t0)
            return df
        except Exception as e:
            log.warning(f"[{name}] {url} недоступен: {e}")
            continue
    REPORT.fail(name, "Оба зеркала недоступны", t0)
    return pd.DataFrame()




## 7. V-DEM — заменён на OWID-зеркало (см. блок 2)

In [ ]:
# 8. BRICS SUMMIT DECLARATIONS
# ----------------------------------------------------------------------------

BRICS_DECLARATION_URLS = {
    2009: "http://www.brics.utoronto.ca/docs/090616-leaders.html",
    2011: "http://www.brics.utoronto.ca/docs/110414-leaders.html",
    2012: "http://www.brics.utoronto.ca/docs/120329-delhi-declaration.html",
    2013: "http://www.brics.utoronto.ca/docs/130327-brics-bank.html",
    2014: "http://www.brics.utoronto.ca/docs/140715-leaders.html",
    2015: "http://www.brics.utoronto.ca/docs/150709-ufa-declaration-ru.pdf",
    2016: "http://www.brics.utoronto.ca/docs/161016-goa.html",
    2017: "http://www.brics.utoronto.ca/docs/170904-xiamen.html",
    2018: "http://www.brics.utoronto.ca/docs/180726-johannesburg.html",
    2019: "http://www.brics.utoronto.ca/docs/191114-brasilia.html",
    2020: "http://www.brics.utoronto.ca/docs/201117-moscow-declaration.html",
    2021: "http://www.brics.utoronto.ca/docs/210909-New-Delhi-Declaration.html",
    2022: "http://www.brics.utoronto.ca/docs/220623-declaration.html",
    2023: "http://brics.utoronto.ca/compliance/2022-beijing-final.pdf",
    2024: "http://www.brics.utoronto.ca/docs/241023-declaration.html",
}

def fetch_brics_declarations() -> pd.DataFrame:
    name = "BRICS Summit Declarations"
    REPORT.start(name)
    t0 = time.time()
    rows = []
    failed_years = []
    for year, url in BRICS_DECLARATION_URLS.items():
        try:
            def _get():
                r = safe_get(url, timeout=30)
                return r.text
            html = with_retries(_get, f"{name}:{year}", retries=2)
            rows.append({"year": year, "url": url, "raw_html_len": len(html), "raw_html": html})
        except Exception as e:
            failed_years.append(year)
            log.debug(f"[{name}] {year} недоступен: {e}")

    if not rows:
        REPORT.fail(name, f"Все {len(BRICS_DECLARATION_URLS)} деклараций недоступны", t0)
        return pd.DataFrame()

    df = pd.DataFrame(rows)
    path = os.path.join(DATA_DIR, "brics_declarations.parquet")
    df.to_parquet(path, index=False)

    if failed_years:
        REPORT.partial(name, df, f"Не загружены годы: {failed_years}", t0)
    else:
        REPORT.success(name, df, path, f"{min(BRICS_DECLARATION_URLS)}-{max(BRICS_DECLARATION_URLS)}", t0=t0)
    return df




## 9. WHO GLOBAL HEALTH OBSERVATORY (GHO) — свободный OData API

In [ ]:

WHO_INDICATORS = {
    "WHOSIS_000001": "life_expectancy_who",
    "WHS4_544": "measles_immunization_pct",
    "NCDMORT3070": "premature_ncd_mortality_pct",
    "SA_0000001688": "alcohol_consumption_per_capita",
    "AIR_11": "pm25_air_pollution_exposure",
    "HIV_0000000001": "hiv_incidence_per_1000",
}


def fetch_who_gho() -> pd.DataFrame:
    name = "WHO Global Health Observatory (GHO)"
    REPORT.start(name)
    t0 = time.time()
    frames = []
    failed = []

    for code, alias in WHO_INDICATORS.items():
        url = f"https://ghoapi.azureedge.net/api/{code}"
        try:
            def _get():
                r = safe_get(url, timeout=30)
                return r.json()
            js = with_retries(_get, f"{name}:{alias}", retries=2)
            recs = js.get("value", [])
            if not recs:
                failed.append(alias)
                continue
            df = pd.DataFrame(recs)
            iso_col = "SpatialDim" if "SpatialDim" in df.columns else None
            year_col = "TimeDim" if "TimeDim" in df.columns else None
            val_col = "NumericValue" if "NumericValue" in df.columns else None
            if not (iso_col and year_col and val_col):
                failed.append(f"{alias}(missing cols)")
                continue
            sub = df[[iso_col, year_col, val_col]].copy()
            sub.columns = ["iso3", "year", alias]
            sub = sub[sub["iso3"].isin(BRICS_ISO3)]
            sub["year"] = pd.to_numeric(sub["year"], errors="coerce")
            sub[alias] = pd.to_numeric(sub[alias], errors="coerce")
            sub = sub.dropna(subset=["year"])
            if not sub.empty:
                frames.append(sub.groupby(["iso3", "year"], as_index=False)[alias].mean())
            else:
                failed.append(f"{alias}(0 после фильтра)")
        except Exception as e:
            failed.append(alias)
            log.debug(f"[{name}] {alias} ({code}) не загружен: {e}")
        time.sleep(0.2)

    if not frames:
        REPORT.fail(name, f"0/{len(WHO_INDICATORS)} индикаторов загружено: {failed[:6]}", t0)
        return pd.DataFrame()

    merged = frames[0]
    for f_ in frames[1:]:
        merged = merged.merge(f_, on=["iso3", "year"], how="outer")

    path = os.path.join(DATA_DIR, "who_gho.parquet")
    merged.to_parquet(path, index=False)

    notes = f"{len(frames)}/{len(WHO_INDICATORS)} индикаторов"
    if failed:
        REPORT.partial(name, merged, f"{notes}; не загружены: {failed}", t0)
    else:
        REPORT.success(name, merged, path, notes=notes, t0=t0)
    return merged




## 10. FAOSTAT — свободный REST API (fenixservices), без ключа

In [ ]:

FAO_ISO3_MAP = {
    "BRA": 21, "RUS": 185, "IND": 100, "CHN": 351, "ZAF": 202,
    "EGY": 59, "ETH": 238, "IDN": 101, "IRN": 102, "ARE": 223
}

FAO_QUERIES = {
    ("FS", "6121", "21010"): "food_supply_kcal_fao",
    ("QCL", "5510", "1717"): "cereal_production_tonnes",
    ("TCL", "5610", "1717"): "cereal_import_tonnes",
}

def fetch_faostat() -> pd.DataFrame:
    """
    REST API fenixservices.fao.org нестабилен (521 Server Error из облачных сред).
    Используем статические bulk-архивы FAOSTAT — стабильный CDN, без ключа.
    """
    name = "FAOSTAT (продовольствие и сельское хозяйство)"
    REPORT.start(name)
    t0 = time.time()

    FAO_BULK_DATASETS = {
        "food_security": "https://fenixservices.fao.org/faostat/static/bulkdownloads/Food_Security_Data_E_All_Data_(Normalized).zip",
        "production_crops": "https://fenixservices.fao.org/faostat/static/bulkdownloads/Production_Crops_Livestock_E_All_Data_(Normalized).zip",
    }

    FAO_ISO3_MAP_NAMES = {
        "Brazil": "BRA", "Russian Federation": "RUS", "India": "IND", "China, mainland": "CHN",
        "South Africa": "ZAF", "Egypt": "EGY", "Ethiopia": "ETH",
        "Indonesia": "IDN", "Iran (Islamic Republic of)": "IRN", "United Arab Emirates": "ARE",
    }

    frames = []
    failed = []

    for alias, url in FAO_BULK_DATASETS.items():
        try:
            def _get():
                r = safe_get(url, timeout=90)
                return r.content
            content = with_retries(_get, f"{name}:{alias}", retries=2)
            zf = zipfile.ZipFile(io.BytesIO(content))
            csv_name = [n for n in zf.namelist() if n.endswith(".csv")][0]
            raw = zf.read(csv_name)
            df = pd.read_csv(io.BytesIO(raw), encoding="latin-1", low_memory=False)

            area_col = next((c for c in df.columns if c.strip() == "Area"), None)
            year_col = next((c for c in df.columns if c.strip() == "Year"), None)
            val_col = next((c for c in df.columns if c.strip() == "Value"), None)
            item_col = next((c for c in df.columns if c.strip() == "Item"), None)
            elem_col = next((c for c in df.columns if c.strip() == "Element"), None)

            if not (area_col and year_col and val_col):
                failed.append(f"{alias}(missing core cols)")
                continue

            df["iso3"] = df[area_col].map(FAO_ISO3_MAP_NAMES)
            sub = df.dropna(subset=["iso3"]).copy()
            if sub.empty:
                failed.append(f"{alias}(0 после фильтра стран)")
                continue

            if alias == "food_security" and item_col and elem_col:
                mask = sub[item_col].astype(str).str.contains("Dietary Energy Supply", case=False, na=False)
                sub = sub[mask]
            elif alias == "production_crops" and item_col and elem_col:
                mask = (sub[item_col].astype(str).str.contains("Cereals", case=False, na=False) &
                        sub[elem_col].astype(str).str.contains("Production", case=False, na=False))
                sub = sub[mask]

            if sub.empty:
                failed.append(f"{alias}(0 после фильтра показателя)")
                continue

            out = sub[["iso3", year_col, val_col]].copy()
            out.columns = ["iso3", "year", alias]
            out["year"] = pd.to_numeric(out["year"], errors="coerce")
            out[alias] = pd.to_numeric(out[alias], errors="coerce")
            out = out.groupby(["iso3", "year"], as_index=False)[alias].mean()
            frames.append(out)
        except Exception as e:
            failed.append(f"{alias}({str(e)[:40]})")
            log.debug(f"[{name}] {alias} не загружен: {e}")

    if not frames:
        REPORT.fail(name, f"0/{len(FAO_BULK_DATASETS)} bulk-архивов загружено: {failed}", t0)
        return pd.DataFrame()

    merged = frames[0]
    for f_ in frames[1:]:
        merged = merged.merge(f_, on=["iso3", "year"], how="outer")

    path = os.path.join(DATA_DIR, "faostat.parquet")
    merged.to_parquet(path, index=False)

    notes = f"{len(frames)}/{len(FAO_BULK_DATASETS)} bulk-архивов"
    if failed:
        REPORT.partial(name, merged, f"{notes}; не загружены: {failed}", t0)
    else:
        REPORT.success(name, merged, path, notes=notes, t0=t0)
    return merged




## 11. FRAGILE STATES INDEX (Fund for Peace) — прямой XLSX по годам

In [ ]:

FSI_COUNTRY_TO_ISO3 = {
    "Brazil": "BRA", "Russia": "RUS", "India": "IND", "China": "CHN",
    "South Africa": "ZAF", "Egypt": "EGY", "Ethiopia": "ETH",
    "Indonesia": "IDN", "Iran": "IRN", "United Arab Emirates": "ARE",
}

FSI_URLS = {
    2019: "https://fragilestatesindex.org/wp-content/uploads/2019/04/fsi-2019.xlsx",
    2020: "https://fragilestatesindex.org/wp-content/uploads/2020/05/fsi-2020.xlsx",
    2021: "https://fragilestatesindex.org/wp-content/uploads/2021/05/fsi-2021.xlsx",
    2022: "https://fragilestatesindex.org/wp-content/uploads/2022/07/fsi-2022-download.xlsx",
}
# World Bank Data Catalog mirror — используется как дополнительный, наиболее свежий источник FSI
FSI_WB_MIRROR_URL = "https://thedocs.worldbank.org/en/doc/cf8eee7ff5029398f75e897b342e7320-0050122023/related/FFP-FSI.xlsx"

def fetch_fragile_states_index() -> pd.DataFrame:
    name = "Fragile States Index (Fund for Peace)"
    REPORT.start(name)
    t0 = time.time()
    frames = []
    failed_years = []

    for year, url in FSI_URLS.items():
        try:
            def _get():
                r = safe_get(url, timeout=40)
                return r.content
            content = with_retries(_get, f"{name}:{year}", retries=2)
            df = pd.read_excel(io.BytesIO(content))
            if "Country" not in df.columns:
                failed_years.append(year)
                continue
            df["iso3"] = df["Country"].map(FSI_COUNTRY_TO_ISO3)
            sub = df.dropna(subset=["iso3"]).copy()
            if sub.empty:
                failed_years.append(year)
                continue
            sub["year"] = year
            score_col = "Total" if "Total" in sub.columns else None
            if score_col is None:
                failed_years.append(f"{year}(no Total col)")
                continue
            keep_cols = ["iso3", "year", score_col]
            frames.append(sub[keep_cols].rename(columns={score_col: "fragility_total_score"}))
        except Exception as e:
            failed_years.append(year)
            log.debug(f"[{name}] {year} не загружен: {e}")

    # Дополнительная попытка: сводный файл-зеркало от World Bank Data Catalog
    # (содержит все годы FSI в одном файле, независимый от fragilestatesindex.org)
    try:
        def _get_wb_mirror():
            r = safe_get(FSI_WB_MIRROR_URL, timeout=40)
            return r.content
        content = with_retries(_get_wb_mirror, f"{name}:WB-mirror", retries=2)
        wb_df = pd.read_excel(io.BytesIO(content), sheet_name=0)
        if "Country" in wb_df.columns and "Year" in wb_df.columns:
            wb_df["iso3"] = wb_df["Country"].map(FSI_COUNTRY_TO_ISO3)
            wb_sub = wb_df.dropna(subset=["iso3"]).copy()
            score_col = "Total" if "Total" in wb_sub.columns else None
            if score_col and not wb_sub.empty:
                wb_sub = wb_sub.rename(columns={"Year": "year", score_col: "fragility_total_score"})
                frames.append(wb_sub[["iso3", "year", "fragility_total_score"]])
    except Exception as e:
        log.debug(f"[{name}] WB-mirror недоступен: {e}")

    if not frames:
        REPORT.fail(name, f"Все {len(FSI_URLS)} лет и WB-зеркало недоступны: {failed_years}", t0)
        return pd.DataFrame()

    full = pd.concat(frames, ignore_index=True)
    full = full.drop_duplicates(subset=["iso3", "year"], keep="first")
    path = os.path.join(DATA_DIR, "fragile_states_index.parquet")
    full.to_parquet(path, index=False)

    notes = f"{len(frames)}/{len(FSI_URLS)} лет загружено"
    if failed_years:
        REPORT.partial(name, full, f"{notes}; не загружены: {failed_years}", t0)
    else:
        REPORT.success(name, full, path, f"{min(FSI_URLS)}-{max(FSI_URLS)}", notes, t0)
    return full




## 12. СБОРКА ЕДИНОГО MASTER-ДАТАСЕТА

In [ ]:

def build_master_dataset(owid_df, wb_df, who_df, fao_df, fsi_df) -> pd.DataFrame:
    frames_available = [(n, d) for n, d in
                         [("owid", owid_df), ("wb", wb_df),
                          ("who", who_df), ("fao", fao_df), ("fsi", fsi_df)]
                         if d is not None and not d.empty]

    if not frames_available:
        log.error("Ни один источник не дал данных — master dataset пуст")
        return pd.DataFrame()

    base_name, master = frames_available[0]
    for n, d in frames_available[1:]:
        d2 = d.copy()
        if "iso3" in d2.columns and "year" in d2.columns:
            dup_cols = [c for c in d2.columns if c in master.columns and c not in ("iso3", "year")]
            d2 = d2.drop(columns=dup_cols) if dup_cols else d2
            master = master.merge(d2, on=["iso3", "year"], how="outer")

    master["is_brics"] = master["iso3"].isin(BRICS_ISO3)
    master = master.sort_values(["iso3", "year"]).reset_index(drop=True)

    path_parquet = os.path.join(OUT_DIR, "brics_master_panel_v6.parquet")
    path_csv = os.path.join(OUT_DIR, "brics_master_panel_v6.csv")
    master.to_parquet(path_parquet, index=False)
    master.to_csv(path_csv, index=False)
    log.info(f"Master dataset v6: {master.shape[0]} строк, {master.shape[1]} колонок -> {path_parquet}")
    return master


def run_pipeline():
    log.info("=" * 60)
    log.info("СТАРТ PIPELINE v6 ЗАГРУЗКИ ДАННЫХ BRICS MARL SC")
    log.info("=" * 60)

    owid_df = fetch_all_owid()
    wb_df = fetch_world_bank()  # содержит NE.TRD.GNFS.ZS — торговля в % ВВП, замена DBnomics
    who_df = fetch_who_gho()
    fao_df = fetch_faostat()
    fsi_df = fetch_fragile_states_index()

    sample_days = ["20220224", "20220301", "20230807", "20240623", "20241023"]
    gdelt_df = fetch_gdelt_events(sample_days)

    unga_df = fetch_unga_votes()
    decl_df = fetch_brics_declarations()

    master = build_master_dataset(owid_df, wb_df, who_df, fao_df, fsi_df)

    REPORT.print_report()
    REPORT.to_json()

    return {
        "owid": owid_df, "world_bank": wb_df,
        "who_gho": who_df, "faostat": fao_df, "fragile_states_index": fsi_df,
        "gdelt": gdelt_df, "unga_votes": unga_df,
        "brics_declarations": decl_df, "master_panel": master,
    }


if __name__ == "__main__":
    results = run_pipeline()
    print("\nГотовые файлы находятся в директориях:")
    print(f"  Сырые данные: ./{DATA_DIR}/")
    print(f"  Обработанный master-датасет: ./{OUT_DIR}/")
    print(f"  Логи и отчёт о загрузке: ./{LOG_DIR}/")



ОТЧЁТ О ЗАГРУЗКЕ ДАННЫХ — BRICS MARL SC PIPELINE v6
Сформирован: 2026-07-07T20:18:57
Источник                                  Статус        Строк   Стран  Время(с)  Комментарий
--------------------------------------------------------------------------------------------------------------
Our World in Data (индикаторы)            OK             3450      10      17.4  23/23 индикаторов загружено
World Bank Open Data (по странам)         OK              300      10      15.7  15/15 индикаторов; неудачных пар: 0
WHO Global Health Observatory (GHO)       OK              250      10      13.3  6/6 индикаторов
FAOSTAT (продовольствие и сельское хозяйство)OK              577      10      14.4  2/2 bulk-архивов
Fragile States Index (Fund for Peace)     OK               40      10       3.0  4/4 лет загружено
GDELT Events (выборка ключевых дат)       OK            67154       0      13.6  5/5 дней загружено
UNGA Voting                               OK           869937       0       2.1  источн

## Запуск полного pipeline

In [ ]:
results = run_pipeline()
print("\nГотовые файлы находятся в директориях:")
print(f"  Сырые данные: ./{DATA_DIR}/")
print(f"  Обработанный master-датасет: ./{OUT_DIR}/")
print(f"  Логи и отчёт о загрузке: ./{LOG_DIR}/")


ОТЧЁТ О ЗАГРУЗКЕ ДАННЫХ — BRICS MARL SC PIPELINE v6
Сформирован: 2026-07-07T20:20:15
Источник                                  Статус        Строк   Стран  Время(с)  Комментарий
--------------------------------------------------------------------------------------------------------------
Our World in Data (индикаторы)            OK             3450      10       9.0  23/23 индикаторов загружено
World Bank Open Data (по странам)         OK              300      10      12.7  15/15 индикаторов; неудачных пар: 0
WHO Global Health Observatory (GHO)       OK              250      10       6.4  6/6 индикаторов
FAOSTAT (продовольствие и сельское хозяйство)OK              577      10      14.0  2/2 bulk-архивов
Fragile States Index (Fund for Peace)     OK               40      10       1.9  4/4 лет загружено
GDELT Events (выборка ключевых дат)       OK            67154       0      13.3  5/5 дней загружено
UNGA Voting                               OK           869937       0       0.8  источн

## Просмотр master-датасета

In [ ]:
import pandas as pd
master = results['master_panel']
print('Форма датасета:', master.shape)
print('\nКолонки:', list(master.columns))
master.head(20)

Форма датасета: (3450, 50)

Колонки: ['iso3', 'year', 'military_spending_gdp', 'military_spending_usd', 'gdp_per_capita', 'gdp_growth', 'population', 'population_growth_rate', 'trade_openness', 'internet_users_pct', 'urbanization', 'life_expectancy', 'co2_emissions', 'press_freedom', 'electoral_democracy_vdem', 'liberal_democracy_vdem', 'political_corruption_vdem', 'corruption_perception', 'unemployment_rate', 'gini_index', 'fdi_inflows', 'electricity_access', 'research_dev_spending', 'human_development_index', 'food_supply_kcal', 'gdp_growth_pct', 'trade_pct_gdp', 'fdi_inflow_pct_gdp', 'population_total', 'dependency_ratio', 'gini_index_wb', 'unemployment_pct_wb', 'military_exp_pct_gdp_wb', 'net_migration_wb', 'internet_users_pct_wb', 'gdp_per_capita_usd', 'inflation_pct', 'gov_debt_pct_gdp', 'urban_pop_pct', 'education_exp_pct_gdp', 'life_expectancy_who', 'measles_immunization_pct', 'premature_ncd_mortality_pct', 'alcohol_consumption_per_capita', 'pm25_air_pollution_exposure', 'hiv_i

,iso3,year,military_spending_gdp,military_spending_usd,gdp_per_capita,gdp_growth,population,population_growth_rate,trade_openness,internet_users_pct,...,life_expectancy_who,measles_immunization_pct,premature_ncd_mortality_pct,alcohol_consumption_per_capita,pm25_air_pollution_exposure,hiv_incidence_per_1000,food_security,production_crops,fragility_total_score,is_brics
0,ARE,-10000,NaN,NaN,NaN,NaN,409.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
1,ARE,-9000,NaN,NaN,NaN,NaN,614.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
2,ARE,-8000,NaN,NaN,NaN,NaN,922.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
3,ARE,-7000,NaN,NaN,NaN,NaN,1383.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
4,ARE,-6000,NaN,NaN,NaN,NaN,2075.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
5,ARE,-5000,NaN,NaN,NaN,NaN,3112.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
6,ARE,-4000,NaN,NaN,NaN,NaN,4668.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
7,ARE,-3000,NaN,NaN,NaN,NaN,7003.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
8,ARE,-2000,NaN,NaN,NaN,NaN,10504.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
9,ARE,-1000,NaN,NaN,NaN,NaN,15757.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True


## Скачивание результатов из Colab

In [ ]:
import shutil
shutil.make_archive('brics_dataset_bundle_v6', 'zip', '.', 'data_processed')
shutil.make_archive('brics_raw_data_v6', 'zip', '.', 'data_raw')
shutil.make_archive('brics_logs_v6', 'zip', '.', 'logs')
from google.colab import files
files.download('brics_dataset_bundle_v6.zip')
files.download('brics_logs_v6.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os, sys, traceback, glob

print("=" * 70)
print("1. ВЕРСИИ БИБЛИОТЕК И PYTHON")
print("=" * 70)
print("Python:", sys.version)
try:
    import pandas as pd
    print("pandas:", pd.__version__)
except Exception as e:
    print("pandas НЕ импортируется:", e)
try:
    import numpy as np
    print("numpy:", np.__version__)
except Exception as e:
    print("numpy НЕ импортируется:", e)
try:
    import openpyxl
    print("openpyxl:", openpyxl.__version__)
except Exception as e:
    print("openpyxl НЕ импортируется (нужен для .xlsx!):", e)

print()
print("=" * 70)
print("2. ТЕКУЩАЯ ДИРЕКТОРИЯ И СПИСОК ФАЙЛОВ")
print("=" * 70)
print("Рабочая директория:", os.getcwd())
print("\nВсе файлы в текущей директории:")
for f in sorted(os.listdir(".")):
    full_path = os.path.join(".", f)
    if os.path.isfile(full_path):
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"  {f}  ({size_mb:.2f} MB)")
    else:
        print(f"  {f}/  [папка]")

print()
print("=" * 70)
print("3. ПОИСК ОЖИДАЕМЫХ ФАЙЛОВ ПО МАСКЕ")
print("=" * 70)
patterns = ["*master_panel*", "*wdi*", "*wgi*", "*.csv", "*.xlsx"]
for pat in patterns:
    matches = glob.glob(f"**/{pat}", recursive=True)
    print(f"Маска '{pat}': найдено {len(matches)} -> {matches[:10]}")

print()
print("=" * 70)
print("4. ПРОВЕРКА КАЖДОГО ОЖИДАЕМОГО ФАЙЛА ПОШТУЧНО")
print("=" * 70)

expected_files = [
    "brics_master_panel_v6_clean.csv",
    "wdi_panel_raw.csv",
    "wgidataset_with_sourcedata-2025.xlsx",
]

for fname in expected_files:
    print(f"\n--- {fname} ---")
    if not os.path.exists(fname):
        print(f"  ФАЙЛ НЕ НАЙДЕН по пути: {os.path.abspath(fname)}")
        continue
    print(f"  Найден, размер: {os.path.getsize(fname)/(1024*1024):.2f} MB")
    try:
        if fname.endswith(".csv"):
            df_test = pd.read_csv(fname, nrows=5, low_memory=False)
            print(f"  Успешно прочитан (первые 5 строк). Колонки ({len(df_test.columns)}):")
            print(" ", list(df_test.columns)[:15])
        elif fname.endswith(".xlsx"):
            xls_test = pd.ExcelFile(fname)
            print(f"  Успешно открыт. Листы ({len(xls_test.sheet_names)}):")
            print(" ", xls_test.sheet_names)
    except Exception as e:
        print(f"  ОШИБКА ПРИ ЧТЕНИИ:")
        traceback.print_exc()

print()
print("=" * 70)
print("5. ПРОВЕРКА НАЛИЧИЯ ФУНКЦИЙ/ПЕРЕМЕННЫХ ИЗ ОСНОВНОГО СКРИПТА")
print("=" * 70)
names_to_check = [
    "integrate_all_sources", "load_and_clean_wdi", "load_and_clean_wgi",
    "MASTER_PANEL_FILE", "WDI_RAW_FILE", "WGI_FILE",
    "BRICS_ISO3", "WDI_SELECTED_INDICATORS", "WGI_INDICATORS",
]
for name in names_to_check:
    exists = name in dir()
    print(f"  {name}: {'определено' if exists else 'НЕ ОПРЕДЕЛЕНО — ячейка с кодом не запускалась?'}")

print()
print("=" * 70)
print("ДИАГНОСТИКА ЗАВЕРШЕНА — скопируйте весь вывод выше и пришлите его")
print("=" * 70)

1. ВЕРСИИ БИБЛИОТЕК И PYTHON
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
pandas: 2.2.2
numpy: 2.0.2
openpyxl: 3.1.5

2. ТЕКУЩАЯ ДИРЕКТОРИЯ И СПИСОК ФАЙЛОВ
Рабочая директория: /content

Все файлы в текущей директории:
  .config/  [папка]
  .ipynb_checkpoints/  [папка]
  brics_master_panel_v6_clean.csv  (0.13 MB)
  brics_master_panel_v7_full.csv  (0.13 MB)
  integration_report.json  (0.00 MB)
  sample_data/  [папка]
  wdi_panel_raw.csv  (2.33 MB)
  wgidataset_with_sourcedata-2025.xlsx  (9.92 MB)

3. ПОИСК ОЖИДАЕМЫХ ФАЙЛОВ ПО МАСКЕ
Маска '*master_panel*': найдено 2 -> ['brics_master_panel_v6_clean.csv', 'brics_master_panel_v7_full.csv']
Маска '*wdi*': найдено 1 -> ['wdi_panel_raw.csv']
Маска '*wgi*': найдено 1 -> ['wgidataset_with_sourcedata-2025.xlsx']
Маска '*.csv': найдено 7 -> ['brics_master_panel_v6_clean.csv', 'wdi_panel_raw.csv', 'brics_master_panel_v7_full.csv', 'sample_data/mnist_train_small.csv', 'sample_data/california_housing_train.csv', 'sample_data/mnist_test.

Список источников (ГОСТ Р 7.0.5-2008 и APA 7th Edition)
1. Our World in Data
ГОСТ: Our World in Data [Электронный ресурс] / Global Change Data Lab. – Оксфорд, 2025. – Режим доступа: https://ourworldindata.org/ (дата обращения: 08.07.2026).

APA: Ritchie, H., Roser, M., Rodés-Guirao, L., & Ortiz-Ospina, E. (2025). Our World in Data. Global Change Data Lab. https://ourworldindata.org/

2. World Bank Open Data / World Development Indicators (WDI)
ГОСТ: World Development Indicators [Электронный ресурс] / The World Bank Group. – Вашингтон, 2025. – Режим доступа: https://databank.worldbank.org/source/world-development-indicators (дата обращения: 08.07.2026).

APA: World Bank. (2025). World Development Indicators. World Bank Group. https://databank.worldbank.org/source/world-development-indicators

3. Worldwide Governance Indicators (WGI)
ГОСТ: Kaufmann, D. The Worldwide Governance Indicators: Methodology and Analytical Issues / D. Kaufmann, A. Kraay, M. Mastruzzi // World Bank Policy Research Working Paper. – 2010. – № 5430. – Режим доступа: https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1682130 (дата обращения: 08.07.2026).

APA: Kaufmann, D., Kraay, A., & Mastruzzi, M. (2010). The Worldwide Governance Indicators: Methodology and analytical issues. World Bank Policy Research Working Paper, (5430). https://papers.ssrn.com/sol3/papers.cfm?abstract_id=1682130

Портал датасета: https://www.worldbank.org/en/publication/worldwide-governance-indicators

4. WHO Global Health Observatory (GHO)
ГОСТ: Global Health Observatory [Электронный ресурс] / World Health Organization. – Женева, 2025. – Режим доступа: https://www.who.int/data/gho (дата обращения: 08.07.2026).

APA: World Health Organization. (2025). Global Health Observatory data repository. https://www.who.int/data/gho

5. FAOSTAT
ГОСТ: FAOSTAT [Электронный ресурс] / Food and Agriculture Organization of the United Nations. – Рим, 2025. – Режим доступа: https://www.fao.org/faostat/ (дата обращения: 08.07.2026).

APA: Food and Agriculture Organization of the United Nations. (2025). FAOSTAT statistical database. https://www.fao.org/faostat/

6. Fragile States Index
ГОСТ: Fragile States Index [Электронный ресурс] / The Fund for Peace. – Вашингтон, 2025. – Режим доступа: https://fragilestatesindex.org/ (дата обращения: 08.07.2026).

APA: The Fund for Peace. (2025). Fragile States Index. https://fragilestatesindex.org/

7. GDELT Project
ГОСТ: The GDELT Project [Электронный ресурс] / GDELT. – 2025. – Режим доступа: https://www.gdeltproject.org/ (дата обращения: 08.07.2026).

APA: GDELT Project. (2025). The GDELT Project: Global Database of Events, Language, and Tone. https://www.gdeltproject.org/

8. UN General Assembly Voting Data
ГОСТ: Voeten, E. Data and Analyses of Voting in the UN General Assembly / E. Voeten // Routledge Handbook of International Organization. – 2013. – Режим доступа: https://github.com/rfordatascience/tidytuesday/tree/main/data/2021/2021-03-23 (дата обращения: 08.07.2026).

APA: Voeten, E. (2013). Data and analyses of voting in the UN General Assembly. In B. Reinalda (Ed.), Routledge handbook of international organization. https://github.com/rfordatascience/tidytuesday/tree/main/data/2021/2021-03-23

9. BRICS Information Centre
ГОСТ: BRICS Information Centre [Электронный ресурс] / University of Toronto, BRICS Research Group. – Торонто, 2025. – Режим доступа: http://www.brics.utoronto.ca/docs/ (дата обращения: 08.07.2026).

APA: University of Toronto, BRICS Research Group. (2025). BRICS Information Centre. http://www.brics.utoronto.ca/docs/

10. V-Dem (Varieties of Democracy)
ГОСТ: Coppedge, M. V-Dem Dataset / M. Coppedge [и др.] // V-Dem Institute. – Гётеборг, 2025. – Режим доступа: https://v-dem.net/data/the-v-dem-dataset/ (дата обращения: 08.07.2026).

APA: Coppedge, M., Gerring, J., Knutsen, C. H., Lindberg, S. I., Teorell, J., et al. (2025). V-Dem dataset (v14). V-Dem Institute. https://v-dem.net/data/the-v-dem-dataset/

In [ ]:
import pandas as pd
import numpy as np
import os
import json

MASTER_PANEL_FILE = "brics_master_panel_v6_clean.csv"
WDI_RAW_FILE = "wdi_panel_raw.csv"
WGI_FILE = "wgidataset_with_sourcedata-2025.xlsx"
OUTPUT_FILE = "brics_master_panel_v7_full.csv"
REPORT_FILE = "integration_report.json"

YEAR_MIN, YEAR_MAX = 1990, 2024
BRICS_ISO3 = ["BRA", "RUS", "IND", "CHN", "ZAF", "EGY", "ETH", "IDN", "IRN", "ARE"]

WGI_INDICATORS = {
    "va": "voice_accountability", "pv": "political_stability",
    "ge": "government_effectiveness", "rq": "regulatory_quality",
    "rl": "rule_of_law", "cc": "control_of_corruption",
}


def load_and_clean_wdi(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        print(f"[WDI] Файл не найден: {filepath} — пропуск")
        return pd.DataFrame()

    raw = pd.read_csv(filepath, low_memory=False)
    print(f"[WDI] Прочитано: {raw.shape}, колонки: {list(raw.columns)}")

    code_col = next((c for c in raw.columns if c.lower() in ("country_code", "iso3", "countrycode")), None)
    year_col = next((c for c in raw.columns if c.lower() == "year"), None)

    if code_col is None or year_col is None:
        print(f"[WDI] Не найдены колонки страны/года. Доступные колонки: {list(raw.columns)}")
        return pd.DataFrame()

    df = raw.copy()
    df = df.rename(columns={code_col: "iso3", year_col: "year"})
    df["iso3"] = df["iso3"].astype(str).str.upper().str.strip()
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    sub = df[df["iso3"].isin(BRICS_ISO3)].copy()
    sub = sub[(sub["year"] >= YEAR_MIN) & (sub["year"] <= YEAR_MAX)]

    drop_cols = [c for c in sub.columns if c.lower() in ("country_name", "countryname", "name")]
    sub = sub.drop(columns=drop_cols, errors="ignore")

    value_cols = [c for c in sub.columns if c not in ("iso3", "year")]
    for c in value_cols:
        sub[c] = pd.to_numeric(sub[c], errors="coerce")
        sub = sub.rename(columns={c: f"{c}_wdi"})

    sub = sub.groupby(["iso3", "year"], as_index=False).mean(numeric_only=True)

    print(f"[WDI] После фильтрации BRICS+{YEAR_MIN}-{YEAR_MAX}: {sub.shape[0]} строк, "
          f"{sub.shape[1]-2} индикаторов")
    return sub


def load_and_clean_wgi(filepath: str) -> pd.DataFrame:
    if not os.path.exists(filepath):
        print(f"[WGI] Файл не найден: {filepath} — пропуск")
        return pd.DataFrame()

    xls = pd.ExcelFile(filepath)
    frames = []

    for code, alias in WGI_INDICATORS.items():
        if code not in xls.sheet_names:
            print(f"[WGI] Лист '{code}' не найден в файле")
            continue

        df_sheet = pd.read_excel(filepath, sheet_name=code, header=0)
        df_sheet.columns = [str(c).strip() for c in df_sheet.columns]

        code_col = "Economy (code)" if "Economy (code)" in df_sheet.columns else None
        if code_col is None:
            code_col = next((c for c in df_sheet.columns
                              if c.lower().strip() == "economy (code)"), None)

        year_col = next((c for c in df_sheet.columns if c.lower() == "year"), None)
        est_col = next((c for c in df_sheet.columns if "governance estimate" in c.lower()), None)

        if code_col is None or year_col is None or est_col is None:
            print(f"[WGI:{code}] Не найдены нужные колонки. Доступно: {list(df_sheet.columns)}")
            continue

        df_sheet[code_col] = df_sheet[code_col].astype(str).str.upper().str.strip()
        df_sheet[year_col] = pd.to_numeric(df_sheet[year_col], errors="coerce")
        df_sheet[est_col] = pd.to_numeric(df_sheet[est_col], errors="coerce")

        sub = df_sheet[
            df_sheet[code_col].isin(BRICS_ISO3)
            & (df_sheet[year_col] >= YEAR_MIN)
            & (df_sheet[year_col] <= YEAR_MAX)
        ][[code_col, year_col, est_col]].copy()

        sub = sub.rename(columns={code_col: "iso3", year_col: "year", est_col: alias})
        sub = sub.groupby(["iso3", "year"], as_index=False)[alias].mean()

        if not sub.empty:
            frames.append(sub)
            print(f"[WGI:{code}] Извлечено {len(sub)} наблюдений")
        else:
            print(f"[WGI:{code}] После фильтра BRICS+год — 0 строк (проверьте код={df_sheet[code_col].unique()[:5]})")

    if not frames:
        print("[WGI] Ни один индикатор не был успешно распарсен")
        return pd.DataFrame()

    merged = frames[0]
    for f in frames[1:]:
        merged = merged.merge(f, on=["iso3", "year"], how="outer")

    print(f"[WGI] Итого: {merged.shape[0]} строк, {merged.shape[1]-2} индикаторов")
    return merged


def integrate_all_sources():
    if not os.path.exists(MASTER_PANEL_FILE):
        raise FileNotFoundError(f"'{MASTER_PANEL_FILE}' не найден")

    master = pd.read_csv(MASTER_PANEL_FILE)
    master["year"] = pd.to_numeric(master["year"], errors="coerce")
    print(f"[MASTER] {master.shape}")

    wdi_df = load_and_clean_wdi(WDI_RAW_FILE)
    wgi_df = load_and_clean_wgi(WGI_FILE)

    combined = master.copy()
    for name, df in [("wdi", wdi_df), ("wgi", wgi_df)]:
        if df is not None and not df.empty:
            dup = [c for c in df.columns if c in combined.columns and c not in ("iso3", "year")]
            df2 = df.drop(columns=dup) if dup else df
            combined = combined.merge(df2, on=["iso3", "year"], how="left")
            print(f"[MERGE {name}] -> {combined.shape}")
        else:
            print(f"[MERGE {name}] пропущен (пусто или файл отсутствует)")

    combined = combined.sort_values(["iso3", "year"]).reset_index(drop=True)
    combined.to_csv(OUTPUT_FILE, index=False)

    report = {
        "rows_final": len(combined), "cols_final": combined.shape[1],
        "wdi_indicators_added": 0 if wdi_df.empty else wdi_df.shape[1]-2,
        "wgi_indicators_added": 0 if wgi_df.empty else wgi_df.shape[1]-2,
        "missing_pct": (combined.isna().mean()*100).round(2).to_dict(),
    }
    with open(REPORT_FILE, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)

    print(f"\nИтог: {combined.shape} -> {OUTPUT_FILE}")
    return combined


result_df = integrate_all_sources()
result_df.head(20)

[MASTER] (350, 38)
[WDI] Прочитано: (6076, 39), колонки: ['country_code', 'country_name', 'year', 'gdp_growth', 'gdp_pc_usd', 'gdp_pc_growth', 'inflation', 'unemployment', 'gross_capital_form', 'govt_consumption', 'exports_gdp', 'imports_gdp', 'trade_gdp', 'fdi_gdp', 'manufacturing_gdp', 'hitech_exports', 'rd_gdp', 'researchers_pm', 'journal_articles', 'patents_residents', 'internet_users', 'mobile_subs', 'broadband', 'secure_servers_pm', 'life_expectancy', 'infant_mortality', 'school_secondary', 'school_tertiary', 'edu_gdp', 'labor_participation', 'female_labor', 'gini_index', 'pop_growth', 'urban_pop_pct', 'renewable_energy', 'electricity_access', 'energy_use_pc', 'forest_area_pct', 'nat_resources_gdp']
[WDI] После фильтрации BRICS+1990-2024: 280 строк, 36 индикаторов
[WGI:va] Извлечено 260 наблюдений
[WGI:pv] Извлечено 260 наблюдений
[WGI:ge] Извлечено 260 наблюдений
[WGI:rq] Извлечено 260 наблюдений
[WGI:rl] Извлечено 260 наблюдений
[WGI:cc] Извлечено 260 наблюдений
[WGI] Итого: 26

,iso3,year,military_spending_usd,gdp_per_capita,population_growth_rate,urbanization,co2_emissions,electoral_democracy_vdem,liberal_democracy_vdem,political_corruption_vdem,...,electricity_access_wdi,energy_use_pc_wdi,forest_area_pct_wdi,nat_resources_gdp_wdi,voice_accountability,political_stability,government_effectiveness,regulatory_quality,rule_of_law,control_of_corruption
0,ARE,1990,NaN,111377.050,5.715,78.685770,51907936.0,0.023,0.055,0.223,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ARE,1991,NaN,106266.195,5.401,78.553860,56895576.0,0.022,0.055,0.223,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ARE,1992,NaN,104206.760,5.102,78.441376,58005216.0,0.025,0.055,0.212,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ARE,1993,NaN,100422.630,4.810,78.351920,65832140.0,0.023,0.055,0.205,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ARE,1994,NaN,102456.100,4.525,78.289130,75350420.0,0.024,0.065,0.135,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ARE,1995,NaN,104527.920,4.421,78.257805,72873750.0,0.027,0.068,0.135,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ARE,1996,NaN,103653.480,8.404,78.268970,75836120.0,0.027,0.070,0.135,...,100.0,11250.169119,3.994058,20.719562,-0.848031,1.053565,0.472621,0.473374,0.626891,0.241391
7,ARE,1997,7.682241e+09,103392.440,7.866,78.382070,73297510.0,0.027,0.070,0.135,...,100.0,10941.746523,4.084779,18.680627,NaN,NaN,NaN,NaN,NaN,NaN
8,ARE,1998,9.058628e+09,96088.490,7.387,78.595550,80624520.0,0.028,0.070,0.135,...,100.0,10375.285993,4.175500,12.527883,-0.844065,0.971102,0.738934,0.483620,0.533521,0.240125
9,ARE,1999,9.365089e+09,92047.590,6.943,78.900270,77844060.0,0.028,0.070,0.135,...,100.0,9839.144035,4.266221,14.854764,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
import pandas as pd
df_va = pd.read_excel("wgidataset_with_sourcedata-2025.xlsx", sheet_name="va", header=0)
df_va.columns = [str(c).strip() for c in df_va.columns]

print("Все колонки листа 'va':")
print(list(df_va.columns))
print()

code_col = next((c for c in df_va.columns if "economy" in c.lower() and "code" in c.lower()), None)
year_col = next((c for c in df_va.columns if c.lower() == "year"), None)
print("Найдена колонка кода:", code_col)
print("Найдена колонка года:", year_col)
print()

print("Уникальные значения колонки кода (первые 30):")
print(df_va[code_col].unique()[:30])
print()

print("Есть ли RUS/CHN/BRA/IND/ZAF в колонке кода?")
for iso in ["RUS", "CHN", "BRA", "IND", "ZAF"]:
    match = df_va[df_va[code_col].astype(str).str.upper().str.strip() == iso]
    print(f"  {iso}: {len(match)} строк")

print()
print("Тип и диапазон значений колонки года:")
print(df_va[year_col].dtype, df_va[year_col].min(), df_va[year_col].max())
print()

print("Пример 10 строк с любой страной BRICS (если есть):")
brics_test = ["RUS", "CHN", "BRA", "IND", "ZAF", "EGY", "ETH", "IDN", "IRN", "ARE"]
sample = df_va[df_va[code_col].astype(str).str.upper().str.strip().isin(brics_test)]
print(sample.head(10).to_string())

Все колонки листа 'va':
['ID variable (economy code/ gov. dimension/ year)', 'Economy (name)', 'Economy (code)', 'Region', 'Income classification', 'Year', 'Governance dimension', 'Number of sources', 'Governance estimate (approx. -2.5 to +2.5)', 'Standard error (estimate)', 'Lower threshold (90% conf. int. estimate)', 'Upper threshold (90% conf. int. estimate)', 'Governance score (0-100)', 'Standard error (gov. score)', 'Lower threshold (90% conf. int. score)', 'Upper threshold (90% conf. int. score)', 'ADB mean', 'AFR mean', 'ARB mean', 'ASB mean', 'ASD mean', 'BPS mean', 'BTI mean', 'CCR mean', 'EBR mean', 'EIU mean', 'EQI mean', 'EUB mean', 'FRH mean', 'GCB mean', 'GCS mean', 'GII mean', 'GWP mean', 'HER mean', 'HRM mean', 'HUM mean', 'IFD mean', 'IJT mean', 'IRP mean', 'LBO mean', 'MSI mean', 'OBI mean', 'PIA mean', 'PRS mean', 'RSF mean', 'VAB mean', 'VDM mean', 'WBS mean', 'WCY mean', 'WJP mean', 'WMO mean']

Найдена колонка кода: ID variable (economy code/ gov. dimension/ year)